## What it compares
Only the important final model families:

- **05** baseline after feature engineering *(included if a summary CSV exists, or via optional manual row)*
- **06** random forest baseline
- **07** best CatBoost / LightGBM notebook
- **10** PCA benchmark
- **11** sampling / SMOTENC benchmark
- **12** proper stacking OOF benchmark
- **13** final 3-model stack with meta-learner comparison
- **14** final 4-model stack with meta-learner comparison

## Final metrics used
We keep only the six core metrics for final reporting:

1. `macro_recall`
2. `recall_2`
3. `precision_2`
4. `ap_class2`
5. `macro_map`
6. `prob_rmse_macro`

## Outputs saved
This notebook saves only compact final outputs into `../models/`:

- `15_final_model_comparison.csv`
- `15_family_winners.csv`
- `15_metric_leaders.csv`
- `15_meta_learner_comparison.csv` *(if available)*
- `15_final_recommendation.md`


In [17]:
# ============================================================
# 1) Paths, final metrics, and source files
# ============================================================
from pathlib import Path
import json
import numpy as np
import pandas as pd

BASE_DIR = Path("..")
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_METRICS = [
    "macro_recall",
    "recall_2",
    "precision_2",
    "ap_class2",
    "macro_map",
    "prob_rmse_macro",
]

# Compact final outputs from this notebook
OUT_COMPARISON = MODEL_DIR / "15_final_model_comparison.csv"
OUT_FAMILY_WINNERS = MODEL_DIR / "15_family_winners.csv"
OUT_METRIC_LEADERS = MODEL_DIR / "15_metric_leaders.csv"
OUT_META_COMPARE = MODEL_DIR / "15_meta_learner_comparison.csv"
OUT_RECOMMENDATION = MODEL_DIR / "15_final_recommendation.md"

# ------------------------------------------------------------
# Source files to compare
# ------------------------------------------------------------
SOURCE_SPECS = [
    {
        "family": "05_baseline_post_fe",
        "label": "05 Baseline Post-FE",
        "files": [
            "05_baseline_post_feature_engineering_summary.csv",
            "05_baseline_post_fe_summary.csv",
            "baseline_post_feature_engineering_final_metrics.csv",
            "baseline_post_fe_summary.csv",
        ],
        "type": "summary",
    },
    {
        "family": "06_random_forest",
        "label": "06 Random Forest",
        "files": [
            "06_random_forest_summary.csv",
            "random_forest_summary.csv",
            "random_forest_final_metrics.csv",
        ],
        "type": "summary",
    },
    {
        "family": "07_best_ml",
        "label": "07 Best CatBoost / LightGBM",
        "files": [
            "07_best_ml_summary.csv",
            "best_ml_summary.csv",
        ],
        "type": "summary",
    },
    {
    "family": "10_pca",
    "label": "10 PCA Benchmark",
    "files": [
        "pca_final_summary.csv",
        "10_pca_final_summary.csv",
        "pca_cv_summary.csv"
    ],
    "type": "summary",
    },
    {
        "family": "11_sampling",
        "label": "11 Sampling / SMOTENC",
        "files": [
            "11_sampling_final_summary.csv",
            "sampling_final_summary.csv",
            "12_sampling_final_summary.csv",
        ],
        "type": "summary",
    },
    {
        "family": "12_stacking_oof",
        "label": "12 Proper Stacking OOF",
        "files": [
            "stacking_2model_final_results.csv",
            "12_stacking_2model_final_results.csv",
            "stacking_2model_cv_summary.csv",
        ],
        "type": "summary",
    },
    {
        "family": "13_stacking_3model",
        "label": "13 Stacking 3-Model",
        "files": [
            "stacking_3model_final_results.csv",
            "13_stacking_3model_final_results.csv",
            "stacking_3model_final_summary.csv",
        ],
        "type": "summary",
    },
    {
        "family": "14_stacking_4model",
        "label": "14 Stacking 4-Model",
        "files": [
            "stacking_4model_final_results.csv",
            "14_stacking_4model_final_results.csv",
            "stacking_4model_final_summary.csv",
        ],
        "type": "summary",
    },
]

# Optional meta-learner comparison CSVs (if produced by notebooks 13 and 14)
META_COMPARE_SPECS = [
    {
        "family": "13_stacking_3model",
        "label": "13 Stacking 3-Model",
        "files": [
            "stacking_3model_meta_compare.csv",
            "13_stacking_3model_meta_compare.csv",
        ],
    },
    {
        "family": "14_stacking_4model",
        "label": "14 Stacking 4-Model",
        "files": [
            "stacking_4model_meta_compare.csv",
            "14_stacking_4model_meta_compare.csv",
        ],
    },
]

# Optional manual rows if a notebook has not yet saved a CSV.
# Leave this empty unless you want to type a summary row manually.
MANUAL_ROWS = []


In [18]:
# ============================================================
# 2) Helper functions
# ============================================================
def find_first_existing_file(candidates, model_dir=MODEL_DIR):
    """Return the first existing file from a list of candidate filenames."""
    for fname in candidates:
        path = model_dir / fname
        if path.exists():
            return path
    return None

def normalize_summary(df, family, family_label, source_file):
    """Normalize different summary CSV formats into one clean comparison schema."""
    out = df.copy()

    # Common renames from older notebooks
    rename_map = {
        "recall_high": "recall_2",
        "precision_high": "precision_2",
        "pred2_rate": "pred_class2_rate",
        "base_model": "model",
        "label": "model",
    }
    out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})

    # If the summary has no explicit model column, create one from the family label.
    if "model" not in out.columns:
        out["model"] = family_label

    # Keep only final / test rows where possible.
    if "split" in out.columns:
        split_str = out["split"].astype(str).str.lower()
        test_rows = out[split_str.str.contains("test", na=False)].copy()
        if len(test_rows):
            out = test_rows
    else:
        model_str = out["model"].astype(str).str.lower()
        test_rows = out[model_str.str.contains("test", na=False)].copy()
        if len(test_rows):
            out = test_rows

    # Ensure the final metrics exist
    for col in FINAL_METRICS:
        if col not in out.columns:
            out[col] = np.nan

    # Keep only the columns we need downstream
    out["family"] = family
    out["family_label"] = family_label
    out["source_file"] = source_file.name if hasattr(source_file, "name") else str(source_file)

    keep_cols = ["family", "family_label", "model", "source_file"] + FINAL_METRICS
    out = out[[c for c in keep_cols if c in out.columns]].copy()

    return out

def load_one_family(spec):
    """Load one model-family summary if its file exists."""
    path = find_first_existing_file(spec["files"])
    if path is None:
        return None

    df = pd.read_csv(path)
    return normalize_summary(df, spec["family"], spec["label"], path)

def rank_table(df):
    """Apply the final ranking logic used for report/poster/dashboard selection."""
    ranked = df.copy()

    ranked = ranked.sort_values(
        by=["macro_recall", "ap_class2", "macro_map", "recall_2", "prob_rmse_macro"],
        ascending=[False, False, False, False, True],
        na_position="last",
    ).reset_index(drop=True)

    if "rank" in ranked.columns:
        ranked = ranked.drop(columns=["rank"])
    ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))
    return ranked

def metric_leader(df, metric, higher_is_better=True):
    usable = df.dropna(subset=[metric]).copy()
    if len(usable) == 0:
        return None

    ascending = not higher_is_better
    usable = usable.sort_values(metric, ascending=ascending).reset_index(drop=True)
    best = usable.iloc[0].copy()
    return {
        "metric": metric,
        "best_family": best["family_label"],
        "best_model": best["model"],
        "best_value": best[metric],
    }


In [19]:

# ============================================================
# 3) Load final summary files
# ============================================================
loaded_tables = []
missing_families = []

for spec in SOURCE_SPECS:
    table = load_one_family(spec)
    if table is None:
        missing_families.append(spec["label"])
    else:
        loaded_tables.append(table)

# Add optional manual rows if supplied
if len(MANUAL_ROWS):
    manual_df = pd.DataFrame(MANUAL_ROWS)
    for col in FINAL_METRICS:
        if col not in manual_df.columns:
            manual_df[col] = np.nan
    needed = ["family", "family_label", "model", "source_file"] + FINAL_METRICS
    loaded_tables.append(manual_df[needed].copy())

if len(loaded_tables) == 0:
    raise FileNotFoundError(
        "No summary files were found in ../models/. Run the model notebooks first, "
        "or add a manual row for Notebook 05 if you want it included."
    )

comparison_df = pd.concat(loaded_tables, ignore_index=True)

print("Loaded model families:")
display(comparison_df[["family_label", "source_file"]].drop_duplicates().reset_index(drop=True))

if missing_families:
    print("\nSkipped because no summary CSV was found:")
    for item in missing_families:
        print("-", item)


Loaded model families:


,family_label,source_file
0,06 Random Forest,random_forest_final_metrics.csv
1,07 Best CatBoost / LightGBM,best_ml_summary.csv
2,10 PCA Benchmark,pca_final_summary.csv
3,11 Sampling / SMOTENC,sampling_final_summary.csv
4,12 Proper Stacking OOF,stacking_2model_final_results.csv
5,13 Stacking 3-Model,stacking_3model_final_results.csv
6,14 Stacking 4-Model,stacking_4model_final_results.csv



Skipped because no summary CSV was found:
- 05 Baseline Post-FE


In [20]:

# ============================================================
# 4) Final model ranking (all loaded candidates)
# ============================================================
final_comparison = rank_table(comparison_df)

if "rank" in final_comparison.columns:
    final_comparison = final_comparison.drop(columns=["rank"])

final_comparison = (
    final_comparison
    .sort_values(
        ["macro_recall", "ap_class2", "macro_map", "recall_2", "prob_rmse_macro"],
        ascending=[False, False, False, False, True]
    )
    .reset_index(drop=True)
)
final_comparison.insert(0, "rank", range(1, len(final_comparison) + 1))

display(final_comparison)

final_comparison.to_csv(OUT_COMPARISON, index=False)
print("Saved:", OUT_COMPARISON)


,rank,family,family_label,model,source_file,macro_recall,recall_2,precision_2,ap_class2,macro_map,prob_rmse_macro
0,1,14_stacking_4model,14 Stacking 4-Model,Stacking_Meta_LogReg,stacking_4model_final_results.csv,0.551059,0.648490,0.068324,0.126966,0.404418,0.390017
1,2,13_stacking_3model,13 Stacking 3-Model,Stacking_Meta_LogReg,stacking_3model_final_results.csv,0.550532,0.640868,0.069351,0.127606,0.404685,0.390424
2,3,07_best_ml,07 Best CatBoost / LightGBM,"CatBoost Two-Threshold (tau1=0.25, tau2=0.12) ...",best_ml_summary.csv,0.550000,0.675169,0.065045,0.123335,0.428370,0.265187
3,4,12_stacking_oof,12 Proper Stacking OOF,Stacking_Meta_LogReg,stacking_2model_final_results.csv,0.549834,0.644093,0.068790,0.127691,0.404709,0.390168
4,5,12_stacking_oof,12 Proper Stacking OOF,CatBoost,stacking_2model_final_results.csv,0.548552,0.629141,0.069430,0.124385,0.403545,0.390769
5,6,13_stacking_3model,13 Stacking 3-Model,CatBoost,stacking_3model_final_results.csv,0.548552,0.629141,0.069430,0.124385,0.403545,0.390769
6,7,14_stacking_4model,14 Stacking 4-Model,CatBoost,stacking_4model_final_results.csv,0.548552,0.629141,0.069430,0.124385,0.403545,0.390769
7,8,10_pca,10 PCA Benchmark,10 LogReg_weather_PCA - Final Test,pca_final_summary.csv,0.548238,0.668426,0.062329,0.117397,0.399952,0.395999
8,9,13_stacking_3model,13 Stacking 3-Model,LogReg,stacking_3model_final_results.csv,0.548120,0.670478,0.062076,0.117800,0.399986,0.395817
9,10,14_stacking_4model,14 Stacking 4-Model,LogReg,stacking_4model_final_results.csv,0.548120,0.670478,0.062076,0.117800,0.399986,0.395817


Saved: ../models/15_final_model_comparison.csv


In [21]:

# ============================================================
# 5) One winner per notebook family
# ============================================================
family_winners = (
    final_comparison
    .sort_values(
        ["macro_recall", "ap_class2", "macro_map", "recall_2", "prob_rmse_macro"],
        ascending=[False, False, False, False, True]
    )
    .drop_duplicates(subset=["family"], keep="first")
    .reset_index(drop=True)
)

if "family_rank" in family_winners.columns:
    family_winners = family_winners.drop(columns=["family_rank"])

family_winners.insert(0, "family_rank", range(1, len(family_winners) + 1))

display(family_winners)

family_winners.to_csv(OUT_FAMILY_WINNERS, index=False)
print("Saved:", OUT_FAMILY_WINNERS)


,family_rank,rank,family,family_label,model,source_file,macro_recall,recall_2,precision_2,ap_class2,macro_map,prob_rmse_macro
0,1,1,14_stacking_4model,14 Stacking 4-Model,Stacking_Meta_LogReg,stacking_4model_final_results.csv,0.551059,0.648490,0.068324,0.126966,0.404418,0.390017
1,2,2,13_stacking_3model,13 Stacking 3-Model,Stacking_Meta_LogReg,stacking_3model_final_results.csv,0.550532,0.640868,0.069351,0.127606,0.404685,0.390424
2,3,3,07_best_ml,07 Best CatBoost / LightGBM,"CatBoost Two-Threshold (tau1=0.25, tau2=0.12) ...",best_ml_summary.csv,0.550000,0.675169,0.065045,0.123335,0.428370,0.265187
3,4,4,12_stacking_oof,12 Proper Stacking OOF,Stacking_Meta_LogReg,stacking_2model_final_results.csv,0.549834,0.644093,0.068790,0.127691,0.404709,0.390168
4,5,8,10_pca,10 PCA Benchmark,10 LogReg_weather_PCA - Final Test,pca_final_summary.csv,0.548238,0.668426,0.062329,0.117397,0.399952,0.395999
5,6,18,06_random_forest,06 Random Forest,"RF Two-Threshold (tau1=0.40, tau2=0.30) - Test",random_forest_final_metrics.csv,0.504606,0.527998,0.053687,0.088000,0.393061,0.376357
6,7,20,11_sampling,11 Sampling / SMOTENC,weight_only - Final Test,sampling_final_summary.csv,0.467248,0.949868,0.024687,0.112772,0.384922,0.515044


Saved: ../models/15_family_winners.csv


In [22]:

# ============================================================
# 6) Metric leaders table
# ============================================================
leader_rows = []

for metric in ["macro_recall", "recall_2", "precision_2", "ap_class2", "macro_map"]:
    row = metric_leader(final_comparison, metric, higher_is_better=True)
    if row is not None:
        leader_rows.append(row)

row = metric_leader(final_comparison, "prob_rmse_macro", higher_is_better=False)
if row is not None:
    leader_rows.append(row)

metric_leaders = pd.DataFrame(leader_rows)
display(metric_leaders)

metric_leaders.to_csv(OUT_METRIC_LEADERS, index=False)
print("Saved:", OUT_METRIC_LEADERS)


,metric,best_family,best_model,best_value
0,macro_recall,14 Stacking 4-Model,Stacking_Meta_LogReg,0.551059
1,recall_2,11 Sampling / SMOTENC,weight_only - Final Test,0.949868
2,precision_2,07 Best CatBoost / LightGBM,CatBoost Argmax - Test,0.173792
3,ap_class2,12 Proper Stacking OOF,Stacking_Meta_LogReg,0.127691
4,macro_map,07 Best CatBoost / LightGBM,CatBoost Argmax - Test,0.428370
5,prob_rmse_macro,07 Best CatBoost / LightGBM,CatBoost Argmax - Test,0.265187


Saved: ../models/15_metric_leaders.csv


In [23]:
# ============================================================
# 7) Optional meta-learner comparison (from notebooks 13 and 14)
# ============================================================
meta_tables = []

for spec in META_COMPARE_SPECS:
    path = find_first_existing_file(spec["files"])
    if path is not None:
        df = pd.read_csv(path).copy()
        df["family"] = spec["family"]
        df["family_label"] = spec["label"]

        # Standardize the expected columns if present
        for col in FINAL_METRICS:
            if col not in df.columns:
                df[col] = np.nan

        keep = ["family", "family_label", "model"] + FINAL_METRICS
        meta_tables.append(df[[c for c in keep if c in df.columns]].copy())

if len(meta_tables):
    meta_compare = pd.concat(meta_tables, ignore_index=True)
    meta_compare = rank_table(meta_compare)
    display(meta_compare)

    meta_compare.to_csv(OUT_META_COMPARE, index=False)
    print("Saved:", OUT_META_COMPARE)
else:
    meta_compare = pd.DataFrame()
    print("No meta-learner comparison files were found.")


,rank,family,family_label,model,macro_recall,recall_2,precision_2,ap_class2,macro_map,prob_rmse_macro
0,1,14_stacking_4model,14 Stacking 4-Model,Meta_LogReg,0.542844,0.586255,0.075015,0.114634,0.402086,0.381583
1,2,13_stacking_3model,13 Stacking 3-Model,Meta_LogReg,0.542041,0.584706,0.075090,0.116107,0.402407,0.382104
2,3,13_stacking_3model,13 Stacking 3-Model,Meta_HistGB,0.340078,0.019602,0.368313,0.116253,0.428245,0.231024
3,4,14_stacking_4model,14 Stacking 4-Model,Meta_HistGB,0.339911,0.018997,0.295627,0.115186,0.427933,0.231015


Saved: ../models/15_meta_learner_comparison.csv


## How to read the tables

Use the outputs like this in your report and poster:

- **Final model ranking** = all loaded candidate models ranked using the final metric logic
- **Family winners** = one best model from each notebook family
- **Metric leaders** = which model is best for each individual metric
- **Meta comparison** = whether `Meta_LogReg` or `Meta_HistGB` was better in the final 3-model / 4-model stacks


In [24]:

# ============================================================
# 8) Final recommendation text for report / poster
# ============================================================
overall_winner = final_comparison.iloc[0]
family_best = family_winners.iloc[0]

recommendation_lines = [
    "# Final model recommendation",
    "",
    f"**Overall winner:** {overall_winner['model']} ({overall_winner['family_label']})",
    "",
    "## Why this model is recommended",
    f"- Macro Recall: {overall_winner.get('macro_recall', np.nan):.6f}" if pd.notna(overall_winner.get('macro_recall', np.nan)) else "- Macro Recall: n/a",
    f"- Recall_2: {overall_winner.get('recall_2', np.nan):.6f}" if pd.notna(overall_winner.get('recall_2', np.nan)) else "- Recall_2: n/a",
    f"- Precision_2: {overall_winner.get('precision_2', np.nan):.6f}" if pd.notna(overall_winner.get('precision_2', np.nan)) else "- Precision_2: n/a",
    f"- AP_Class2: {overall_winner.get('ap_class2', np.nan):.6f}" if pd.notna(overall_winner.get('ap_class2', np.nan)) else "- AP_Class2: n/a",
    f"- Macro MAP: {overall_winner.get('macro_map', np.nan):.6f}" if pd.notna(overall_winner.get('macro_map', np.nan)) else "- Macro MAP: n/a",
    f"- Prob RMSE Macro: {overall_winner.get('prob_rmse_macro', np.nan):.6f}" if pd.notna(overall_winner.get('prob_rmse_macro', np.nan)) else "- Prob RMSE Macro: n/a",
    "",
]

if len(meta_compare):
    best_meta = meta_compare.iloc[0]
    recommendation_lines += [
        "",
        "## Meta-learner note",
        f"- Best meta-learner result observed: {best_meta['model']} ({best_meta['family_label']})",
    ]

recommendation_text = "\n".join(recommendation_lines)
print(recommendation_text)

with open(OUT_RECOMMENDATION, "w", encoding="utf-8") as f:
    f.write(recommendation_text)

print("\nSaved:", OUT_RECOMMENDATION)


# Final model recommendation

**Overall winner:** Stacking_Meta_LogReg (14 Stacking 4-Model)

## Why this model is recommended
- Macro Recall: 0.551059
- Recall_2: 0.648490
- Precision_2: 0.068324
- AP_Class2: 0.126966
- Macro MAP: 0.404418
- Prob RMSE Macro: 0.390017


## Meta-learner note
- Best meta-learner result observed: Meta_LogReg (14 Stacking 4-Model)

Saved: ../models/15_final_recommendation.md
